# 10 — Red Team Operations and Automation

## What This Is
Red teaming simulates real-world adversary behaviour across the full kill chain to test an organisation's detection and response capabilities. Unlike penetration testing (finds vulns), red teaming tests whether defenders detect and respond effectively.

## Why It Matters
Blue teams optimise for the threats they know. Red teams expose the threats they don't. Mature organisations run continuous red team exercises (Purple Team, Breach & Attack Simulation) to validate detection logic.

## Where the Community Stands
MITRE ATT&CK Navigator maps red team coverage. BAS (Breach & Attack Simulation) platforms (SafeBreach, AttackIQ) automate TTP emulation. VECTR tracks red/blue engagement data.

## Authorized Testing Context
All operation planning and simulation here is for authorised engagements only. Red team operations require a signed Rules of Engagement (RoE) document explicitly scoping targets, timeframes, and allowed techniques. Never perform any of these actions without written authorisation.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import random, hashlib

# Kill chain phases (Lockheed Martin Cyber Kill Chain)
KILL_CHAIN = [
    'Reconnaissance',
    'Weaponisation',
    'Delivery',
    'Exploitation',
    'Installation',
    'Command & Control',
    'Actions on Objectives',
]

# Unified Kill Chain (more granular, 18 phases)
UNIFIED_PHASES = [
    'Reconnaissance', 'Weaponisation', 'Delivery', 'Social Engineering',
    'Exploitation', 'Persistence', 'Defence Evasion', 'C2',
    'Pivoting', 'Discovery', 'Privilege Escalation', 'Execution',
    'Credential Access', 'Lateral Movement', 'Collection', 'Exfiltration',
    'Impact', 'Objectives',
]

print('Cyber Kill Chain:')
for i, phase in enumerate(KILL_CHAIN, 1):
    print(f'  Phase {i}: {phase}')

In [ ]:
@dataclass
class EngagementScope:
    client:          str
    start_date:      str
    end_date:        str
    in_scope_ranges: List[str]
    excluded_systems:List[str]
    allowed_ttps:    List[str]
    prohibited_ttps: List[str]
    escalation_contacts: List[str]

@dataclass
class OperationStep:
    phase:        str
    technique_id: str
    description:  str
    target:       str
    status:       str = 'PLANNED'   # PLANNED/EXECUTED/DETECTED/BLOCKED
    notes:        str = ''

class RedTeamOp:
    def __init__(self, scope: EngagementScope):
        self.scope = scope
        self.steps: List[OperationStep] = []
        self.detections: List[str] = []

    def add_step(self, step: OperationStep) -> bool:
        # Validate against RoE
        if step.technique_id in self.scope.prohibited_ttps:
            print(f'  [RoE BLOCK] {step.technique_id} is prohibited — skipping')
            return False
        if step.technique_id not in self.scope.allowed_ttps:
            print(f'  [RoE WARN] {step.technique_id} not explicitly in-scope — confirm before executing')
        self.steps.append(step)
        return True

    def execute_step(self, step: OperationStep, detected: bool = False, blocked: bool = False):
        if blocked:
            step.status = 'BLOCKED'
        elif detected:
            step.status = 'DETECTED'
            self.detections.append(step.technique_id)
        else:
            step.status = 'EXECUTED'

    def coverage_report(self) -> Dict:
        total     = len(self.steps)
        executed  = sum(1 for s in self.steps if s.status in ('EXECUTED','DETECTED'))
        detected  = sum(1 for s in self.steps if s.status == 'DETECTED')
        blocked   = sum(1 for s in self.steps if s.status == 'BLOCKED')
        detection_rate = detected / executed if executed > 0 else 0.0
        return {
            'total_steps':     total,
            'executed':        executed,
            'detected':        detected,
            'blocked':         blocked,
            'detection_rate':  round(detection_rate, 3),
            'undetected_ttps': [s.technique_id for s in self.steps
                                if s.status == 'EXECUTED'],
        }

scope = EngagementScope(
    client='ACME Corp', start_date='2024-03-01', end_date='2024-03-15',
    in_scope_ranges=['10.0.1.0/24', '10.0.2.0/24'],
    excluded_systems=['10.0.1.1', 'prod-payment-server'],
    allowed_ttps=['T1566.001','T1078','T1059','T1055','T1041','T1003','T1021','T1486'],
    prohibited_ttps=['T1499','T1485'],   # No DoS, no wipe
    escalation_contacts=['ciso@acme.com', 'soc-lead@acme.com']
)

op = RedTeamOp(scope)

# Plan operation steps
steps = [
    OperationStep('Delivery',      'T1566.001', 'Spear phishing to finance team',     'finance@acme.com'),
    OperationStep('Installation',  'T1078',     'Use harvested credentials for VPN',  'vpn.acme.com'),
    OperationStep('C2',            'T1059',     'PowerShell beacon to C2',             '10.0.1.50'),
    OperationStep('Privilege Esc', 'T1055',     'Process injection into lsass',        '10.0.1.50'),
    OperationStep('Cred Access',   'T1003',     'Credential dumping (Mimikatz-style)',  '10.0.1.50'),
    OperationStep('Lat Movement',  'T1021',     'Pass-the-hash lateral movement',      '10.0.1.20'),
    OperationStep('Exfiltration',  'T1041',     'Data exfil over HTTPS C2 channel',    'external'),
    OperationStep('Prohibited',    'T1499',     'DDoS — should be blocked by RoE',    'all'),
]

for step in steps:
    op.add_step(step)

print('\nSimulated execution results:')

In [ ]:
# Simulate execution with detection outcomes
detection_outcomes = [
    ('T1566.001', False, False),   # Phishing: undetected
    ('T1078',     True,  False),   # VPN login: detected (geo anomaly)
    ('T1059',     False, False),   # PowerShell: evaded EDR
    ('T1055',     True,  True),    # lsass injection: detected AND blocked
    ('T1003',     False, False),   # Cred dump: undetected
    ('T1021',     True,  False),   # Lateral: detected
    ('T1041',     False, False),   # Exfil: undetected
]

for step in op.steps:
    for ttp, detected, blocked in detection_outcomes:
        if step.technique_id == ttp:
            op.execute_step(step, detected, blocked)

report = op.coverage_report()
print('=== Red Team Engagement Summary ===')
for k, v in report.items():
    print(f'  {k}: {v}')

print('\nStep-by-step results:')
STATUS_EMOJI = {'EXECUTED': '[PASS]', 'DETECTED': '[DETECTED]', 'BLOCKED': '[BLOCKED]', 'PLANNED': '[PLANNED]'}
for s in op.steps:
    label = STATUS_EMOJI.get(s.status, s.status)
    print(f'  {label:<12} {s.technique_id:<12} {s.description}')

## Purple Team Exercises
Purple teaming is collaborative: red team executes a TTP while blue team simultaneously tries to detect it. This accelerates detection engineering rather than treating red/blue as adversarial.

In [ ]:
@dataclass
class PurpleExercise:
    ttp_id:        str
    ttp_name:      str
    executed_at:   str
    detection_fired: bool
    alert_time_sec: Optional[float]   # None if not detected
    log_source:    Optional[str]
    remediation:   str

def purple_team_report(exercises: List[PurpleExercise]) -> None:
    total     = len(exercises)
    detected  = sum(1 for e in exercises if e.detection_fired)
    missed    = total - detected
    avg_time  = sum(e.alert_time_sec for e in exercises
                    if e.alert_time_sec) / max(1, detected)

    print('=== Purple Team Exercise Report ===')
    print(f'  Total exercises:  {total}')
    print(f'  Detected:         {detected} ({detected/total*100:.0f}%)')
    print(f'  Missed:           {missed}')
    print(f'  Avg alert time:   {avg_time:.1f}s')
    print('\n  Missed detections (improve these rules):')
    for e in exercises:
        if not e.detection_fired:
            print(f'    {e.ttp_id} {e.ttp_name} — {e.remediation}')

exercises = [
    PurpleExercise('T1566.001','Phishing',    '09:00', True,  45.0, 'Email Gateway', 'Tune attachment sandbox'),
    PurpleExercise('T1059',    'PowerShell',  '10:30', False, None, None,            'Create EDR rule: encoded -enc flag'),
    PurpleExercise('T1003',    'Cred Dump',   '11:00', False, None, None,            'Alert on lsass handle open events'),
    PurpleExercise('T1041',    'Exfiltration','14:00', True,  120.0,'Network IDS',   'Reduce threshold for large HTTPS uploads'),
    PurpleExercise('T1021',    'Pass-Hash',   '15:30', True,  30.0, 'SIEM',          'Alert working well — tune noise reduction'),
]

purple_team_report(exercises)